# 03 - Create a Square YOLO ROI Dataset

This notebook uses the full bilateral PNG images exported by notebook 01 and your YOLOv8 checkpoint. It writes a classifier dataset with the same simple folder layout as the original dataset:

`train/<KL grade>/image.png` and `val/<KL grade>/image.png`.

Each YOLO box is expanded by 1.15x. The shorter dimension is extended to make a square. When the square extends outside the radiograph, only the missing area is padded black. No center crop is used.


In [ ]:
!pip -q install "ultralytics>=8.3,<9"


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import math
from pathlib import Path

import cv2
import numpy as np
import torch
from ultralytics import YOLO

DATA_ROOT = Path("/content/drive/MyDrive/Datasets/KneeXrayData_Mendeley_v1/extracted/KneeXrayData")
LABEL_ROOT = DATA_ROOT / "ClsKLData/kneeKL224"
FULL_IMAGE_ROOT = Path(
    "/content/drive/MyDrive/Datasets/KneeXrayData_Mendeley_v1/derived/"
    "full_bilateral_png_v1"
)
YOLO_CHECKPOINT = Path("/content/drive/MyDrive/Models/yolov8_checkpoint/best.pt")
OUTPUT_ROOT = Path(
    "/content/drive/MyDrive/Datasets/KneeXrayData_Mendeley_v1/derived/"
    "densenet121_yolo_square_roi_trainval_v1"
)

if OUTPUT_ROOT.exists():
    raise FileExistsError(
        f"ROI dataset already exists: {OUTPUT_ROOT}. Use it, or choose a new name."
    )
for required in (LABEL_ROOT, FULL_IMAGE_ROOT, YOLO_CHECKPOINT):
    if not required.exists():
        raise FileNotFoundError(required)


## Read the existing KL labels

The filename already contains the patient identifier and side, for example `9011115R.png`. YOLO only supplies the ROI; it never supplies the KL label.


In [ ]:
def patient_and_side(path):
    stem = Path(path).stem
    return stem[:-1], stem[-1].upper()


labels = {}
for split in ("train", "val"):
    for grade in range(5):
        for image_path in (LABEL_ROOT / split / str(grade)).glob("*.png"):
            patient, side = patient_and_side(image_path)
            labels[(split, patient, side)] = grade

print("Labels found:", len(labels))


## Define square ROI geometry

The ROI is not resized here. The training notebook performs its usual single resize to 384x384.


In [ ]:
BOX_EXPANSION = 1.15


def make_square_roi(image, xyxy):
    image_height, image_width = image.shape[:2]
    x1, y1, x2, y2 = map(float, xyxy)
    box_width, box_height = x2 - x1, y2 - y1
    if box_width <= 0 or box_height <= 0:
        raise ValueError(f"Invalid YOLO box: {xyxy}")

    center_x = (x1 + x2) / 2
    center_y = (y1 + y2) / 2
    side = int(math.ceil(max(box_width, box_height) * BOX_EXPANSION))
    wanted_x1 = int(math.floor(center_x - side / 2))
    wanted_y1 = int(math.floor(center_y - side / 2))
    wanted_x2 = wanted_x1 + side
    wanted_y2 = wanted_y1 + side

    crop = image[
        max(0, wanted_y1):min(image_height, wanted_y2),
        max(0, wanted_x1):min(image_width, wanted_x2),
    ]
    return cv2.copyMakeBorder(
        crop,
        max(0, -wanted_y1),
        max(0, wanted_y2 - image_height),
        max(0, -wanted_x1),
        max(0, wanted_x2 - image_width),
        cv2.BORDER_CONSTANT,
        value=(0, 0, 0),
    )


## Run YOLO and save the train/validation ROIs

For each bilateral radiograph, the two most confident detections are ordered from left to right in the displayed image and saved as `R` then `L`, matching the existing project convention.


In [ ]:
OUTPUT_ROOT.mkdir(parents=True)
for split in ("train", "val"):
    for grade in range(5):
        (OUTPUT_ROOT / split / str(grade)).mkdir(parents=True)

detector = YOLO(str(YOLO_CHECKPOINT))
device = 0 if torch.cuda.is_available() else "cpu"

for split in ("train", "val"):
    image_paths = sorted((FULL_IMAGE_ROOT / split).glob("*.png"))
    if not image_paths:
        raise FileNotFoundError(f"No full PNG images found: {FULL_IMAGE_ROOT / split}")

    for index, image_path in enumerate(image_paths, start=1):
        image = cv2.imread(str(image_path), cv2.IMREAD_COLOR)
        if image is None:
            raise RuntimeError(f"Cannot read full PNG image: {image_path}")
        result = detector.predict(
            image,
            conf=0.45,
            imgsz=640,
            device=device,
            verbose=False,
        )[0]

        boxes = result.boxes.xyxy.detach().cpu().numpy()
        scores = result.boxes.conf.detach().cpu().numpy()
        boxes = boxes[np.argsort(scores)[::-1][:2]]
        boxes = sorted(boxes, key=lambda box: float(box[0] + box[2]))
        if len(boxes) != 2:
            raise RuntimeError(f"Expected two knee detections: {image_path}")

        patient = image_path.stem
        for box, side in zip(boxes, ("R", "L")):
            grade = labels.get((split, patient, side))
            if grade is None:
                raise RuntimeError(f"Missing KL label: {split}/{patient}{side}")
            roi = make_square_roi(image, box)
            destination = OUTPUT_ROOT / split / str(grade) / f"{patient}{side}.png"
            if not cv2.imwrite(str(destination), roi):
                raise RuntimeError(f"Cannot write {destination}")

        if index % 100 == 0 or index == len(image_paths):
            print(f"{split}: {index}/{len(image_paths)} bilateral images")

print("ROI dataset created:", OUTPUT_ROOT)
print("Next: run notebook 04.")
